# Diagnostic visuel des faux négatifs / faux positifs

- **FN** : chirp annoté mais non associé à une détection.
- **FP** : détection sans chirp manuel associé.

Le spectrogramme superpose annotations et détections proches.


In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from tkinter import Tk, filedialog
import pandas as pd
from benchmark import run_benchmark
from benchmark.visual_diagnostics import build_error_cases, error_case_summary, plot_error_case


## Localiser le projet


In [ ]:
def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "bat_analysis").exists():
            return candidate
    raise FileNotFoundError("Impossible de trouver la racine du projet")
PROJECT_ROOT = find_project_root()
analysis_py = PROJECT_ROOT / "bat_analysis" / "modelling.py"
print(PROJECT_ROOT)
print(analysis_py)


## Choisir le JSON


In [ ]:
root = Tk(); root.withdraw(); root.attributes("-topmost", True)
annotation_json = filedialog.askopenfilename(title="Sélectionner bat_chirp_annotations.json", filetypes=[("JSON","*.json")])
root.destroy()
annotation_json = Path(annotation_json)
print(annotation_json)


## Benchmark courant


In [ ]:
result = run_benchmark(annotation_json=annotation_json, analysis_py=analysis_py, min_iou=0.05, max_center_error_ms=4.0, verbose=True)
pd.Series(result.summary)


## Liste FP/FN


In [ ]:
cases = build_error_cases(result)
display(error_case_summary(cases))
display(cases)
fn_cases = cases[cases["case_type"]=="FN"].reset_index(drop=True)
fp_cases = cases[cases["case_type"]=="FP"].reset_index(drop=True)


## Visualiser un cas

Rouge = annotation FN ciblée, orange = autres annotations, cyan = FP ciblé, blanc = autres détections.


In [ ]:
case_id = fn_cases.iloc[0]["case_id"] if len(fn_cases) else fp_cases.iloc[0]["case_id"]
fig = plot_error_case(result, annotation_json, case_id, cases=cases, window_ms=40.0, fmin_khz=20, fmax_khz=150, db_floor=-90)
fig.show()


## Navigation rapide


In [ ]:
def show_fn(index, window_ms=40):
    row = fn_cases.iloc[index]
    print(row.to_string())
    return plot_error_case(result, annotation_json, row, cases=cases, window_ms=window_ms, db_floor=-90)

def show_fp(index, window_ms=40):
    row = fp_cases.iloc[index]
    print(row.to_string())
    return plot_error_case(result, annotation_json, row, cases=cases, window_ms=window_ms, db_floor=-90)

# show_fn(0).show()
# show_fp(0).show()


## Détections proches d'un FN


In [ ]:
def nearby_detections(case_id, radius_ms=10.0):
    row = cases[cases["case_id"] == case_id].iloc[0]
    rel = row["relative_path"]; center = float(row["center_ms"])
    d = result.detections[result.detections["relative_path"] == rel].copy()
    d["distance_to_case_ms"] = (d["time_mid_ms"] - center).abs()
    return d[d["distance_to_case_ms"] <= radius_ms].sort_values("distance_to_case_ms")

# nearby_detections("FN001", radius_ms=10)


## Classification manuelle optionnelle


In [ ]:
review = cases.copy()
review["diagnostic_label"] = ""
review["comment"] = ""
review


In [ ]:
output_dir = Path(annotation_json).parent / "benchmark_results" / "visual_fp_fn"
output_dir.mkdir(parents=True, exist_ok=True)
review.to_csv(output_dir / "fp_fn_manual_review.csv", index=False)
print(output_dir / "fp_fn_manual_review.csv")
